### **01 - Data Preparation and Sequencing**
#### Credit Card Fraud Detection Pipeline

#### **Overview**
This notebook performs data preprocessing and feature engineering for a credit card fraud detection system. The pipeline includes data cleaning, feature engineering, imbalanced data handling, and sequence preparation for LSTM modeling.

#### **Key Steps:**

- Load and explore transaction data
- Filter relevant transaction types
- Engineer balance error features
- Handle class imbalance through stratified sampling
- Scale features and prepare sequences for temporal modeling
- Export processed datasets for model training

### Environment Setup

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler
import seaborn as sns
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

### Configuration

In [ ]:
# Global parameters
SEQ_LEN = 1  # Sequence length for LSTM (1 = single timestep)
TEST_SIZE = 0.2
RANDOM_STATE = 42
FRAUD_RATIO = 5  # Number of clean samples per fraud case

In [ ]:
le = LabelEncoder()

In [ ]:
df = pd.read_csv("/content/PS_20174392719_1491204439457_log.csv")

Initial Data Exploration

In [ ]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [ ]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


####**Data Filtering**
Fraud primarily occurs in two transaction types: TRANSFER and CASH_OUT. We filter the dataset to focus on these high-risk categories.

In [ ]:
print (f"Original shape of the dataset{df.shape}")

Original shape of the dataset(6362620, 11)


In [ ]:

df = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])]
print(f" Shape after filtering: {df.shape}")

 Shape after filtering: (2770409, 11)


### **Feature Engineering**
####**Balance Error Detection**
We create features to detect discrepancies in account balances, which are strong indicators of fraudulent activity.

In [ ]:
df['errorBalanceOrig'] = df['newbalanceOrig'] + df['amount'] - df['oldbalanceOrg']
df['errorBalanceDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']

In [ ]:
df['type_enc'] = le.fit_transform(df['type'])

In [ ]:
fraud_df = df[df['isFraud'] == 1]
clean_df = df[df['isFraud'] == 0]

In [ ]:
print(f"length {len(fraud_df)} and {len(clean_df)}  ")

length 8213 and 2762196  


In [ ]:
clean_sampled = clean_df.sample(n=len(fraud_df) * 5, random_state=42)

In [ ]:
combined_df = pd.concat([fraud_df, clean_sampled]).sample(frac=1, random_state=42)
print(f"Balanced Dataset Shape: {combined_df.shape}")
print(f"Fraud Count: {len(fraud_df)}")

Balanced Dataset Shape: (49278, 14)
Fraud Count: 8213


In [ ]:
features = ['step', 'type_enc', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
            'oldbalanceDest', 'newbalanceDest', 'errorBalanceOrig', 'errorBalanceDest']

In [ ]:
X_raw = combined_df[features].values
y_raw = combined_df['isFraud'].values

In [ ]:
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_raw)


In [ ]:
X_final = X_scaled.reshape((X_scaled.shape[0], 1, X_scaled.shape[1]))
y_final = y_raw

In [ ]:
print(f" Final X Shape: {X_final.shape}")
print(f" Final y Shape: {y_final.shape}")

 Final X Shape: (49278, 1, 9)
 Final y Shape: (49278,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.2, random_state=42, stratify=y_final)


np.save('X_train.npy', X_train)
np.save('X_test.npy', X_test)
np.save('y_train.npy', y_train)
np.save('y_test.npy', y_test)


In [ ]:
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('columns.pkl', 'wb') as f:
    pickle.dump(features, f)

 Data Prep Complete. Artifacts saved!
